In [1]:
import csv
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from PIL import Image

In [2]:
filenames = [
    'trace_mutex_timeline_1_CPU_results.txt', 
    'trace_mutex_timeline_2_CPU_results.txt', 
    'trace_mutex_timeline_3_CPU_results.txt', 
    'trace_mutex_timeline_5_CPU_results.txt', 
    'trace_mutex_timeline_all_CPU_results.txt'
]
file_index = 4
cpu_count = '16'

In [3]:
start = 0
forks = dict()
exits = dict()
tids = list()
locks = dict()
unlocks = dict()

with open(filenames[file_index], 'r') as file:
    for og_line in file:
        line = og_line.strip()
        if line == '':
            continue
        args = [int(arg) for arg in line.split(',')]
        tid = args[2]
        if start == 0:
            start = args[0]
        time = args[0] - start
        match args[1]:
            case 0: # ret:lock
                locks[tid].append(time)
            case 1: # ent:unlock
                unlocks[tid].append(time)
            case 2: # fork
                tids.append(tid)
                forks[tid] = time
                locks[tid] = []
                unlocks[tid] = []
            case 3: # exit
                exits[tid] = time
            case _:
                print(args)
                break

In [4]:
# sanity check: all unlocks happen after locks in each thread
for tid in tids:
    lock_times = locks[tid]
    unlock_times = unlocks[tid]
    if len(lock_times) != len(unlock_times):
        print('length mismatch')
        break
    for i in range(len(lock_times)):
        if lock_times[i] > unlock_times[i]:
            print(lock_times[i], unlock_times[i], tid)
            break

In [5]:
durations = dict()
for tid in tids:
    durations[tid] = [unlocks[tid][i] - locks[tid][i] for i in range(len(locks[tid]))]

fig, ax = plt.subplots(figsize=(10,5))

ypos = 0

for tid in tids:
    ax.barh(y=ypos, width=durations[tid], left=locks[tid], height=0.4, color='skyblue', edgecolor='black')
    ypos += 1

ax.set_yticks(range(len(tids)))
ax.set_yticklabels(tids)
ax.grid(axis='x', linestyle='--', alpha=0.7)

ax.set_title(f'Mutex Hold timeline ({cpu_count} CPU)')
ax.set_ylabel('tid')
ax.set_xlabel('time since program starts (ns)')

plt.savefig('plots/mutex_hold_timeline_' + cpu_count + '.png', dpi=300)
plt.close()